# SplinePLSI and NeuralPLSI

This notebook fits `SplinePLSI` (a B-spline link) and `NeuralPLSI` (a neural-network link) to simulated data and compares the estimates with the truth.

**Scenario:**
- **n**: 500
- **Outcome**: continuous
- **Link g**: sigmoid
- **Exposure distribution**: normal

Besides the package, the notebook needs matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from neuralplsi import NeuralPLSI, SplinePLSI
from neuralplsi.simulation import simulate_data, beta as true_beta, gamma as true_gamma

plt.rcParams['axes.grid'] = True

## 1. Data generation

The outcome $Y$ depends on the exposures $X$ through a single index $\beta^T X$, transformed by a nonlinear link $g(\cdot)$, plus linear covariate effects $Z\gamma$.

In [ ]:
n = 500
outcome = 'continuous'
g_fn_name = 'sigmoid'
seed = 42

X, Z, y, xb, gxb, true_g_fn = simulate_data(n, outcome=outcome, g_type=g_fn_name, seed=seed)

print(f"X shape: {X.shape}")
print(f"Z shape: {Z.shape}")
print(f"y shape: {y.shape}")

## 2. Model fitting

Both estimators fit in the package's C++ engine. `random_state` seeds the starting values, so the fits are reproducible.

In [ ]:
spline = SplinePLSI(family=outcome).fit(X, Z, y, random_state=0)
neural = NeuralPLSI(family=outcome).fit(X, Z, y, random_state=0)

print("True gamma:      ", np.round(true_gamma, 3))
print("SplinePLSI gamma:", np.round(spline.gamma, 3))
print("NeuralPLSI gamma:", np.round(neural.gamma, 3))

## 3. Estimated links

The estimated links $g(\cdot)$ against the true link, over the range of the true index.

In [ ]:
grid = np.linspace(xb.min(), xb.max(), 100)
true_g_vals = true_g_fn(grid)
spline_g_vals = spline.g_function(grid)
neural_g_vals = neural.g_function(grid)

plt.figure(figsize=(10, 6))
plt.plot(grid, true_g_vals, 'k--', label='True g', linewidth=2)
plt.plot(grid, spline_g_vals, 'b-', label='SplinePLSI', alpha=0.7)
plt.plot(grid, neural_g_vals, 'r-', label='NeuralPLSI', alpha=0.7)
plt.title(f"Estimated and true g ({g_fn_name})", fontsize=14)
plt.xlabel("Index (beta^T X)")
plt.ylabel("g(index)")
plt.legend()
plt.show()

### Index coefficients

The estimated beta against the true beta (both have unit length and `beta[0] >= 0`).

In [ ]:
plt.figure(figsize=(10, 5))
indices = np.arange(len(true_beta))
width = 0.25

plt.bar(indices - width, true_beta, width, label='True', color='black', alpha=0.6)
plt.bar(indices, spline.beta, width, label='SplinePLSI', color='blue', alpha=0.6)
plt.bar(indices + width, neural.beta, width, label='NeuralPLSI', color='red', alpha=0.6)

plt.xlabel("Exposure")
plt.ylabel("beta")
plt.title("Index coefficients")
plt.legend()
plt.show()